# Azzaro: Whisper turbo vs small vs large

Comparamos tres modelos de Whisper sobre el mismo video, contamos diferencias de palabras y revisamos los clips donde no coinciden. El caso puntual a mirar es `racingista` / `racinguista`.

## 1. Configuracion

In [1]:
import base64
from itertools import combinations
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import HTML, Markdown, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    buscar_ffmpeg,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    normalizar_texto,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None  # si ya tenes el mp4 local, poner aca la ruta y dejar VIDEO_URL como esta

MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"  # poner None si YouTube no pide login
FORCE_TRANSCRIBE = False

CLIP_MARGIN_SECONDS = 0.0  # 0.0 muestra el intervalo real usado por la transcripcion original
MAX_CASOS_A_MOSTRAR = None  # poner un numero si queres limitar la revision visual

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR


WindowsPath('C:/Users/bianc/NLP Natural Language Processing/labios-argentos/evaluation/outputs/azzaro_whisper')

## 2. Descargar o cargar video

In [2]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

WindowsPath('C:/Users/bianc/NLP Natural Language Processing/labios-argentos/evaluation/outputs/azzaro_whisper/videos/azzaro_racing_caracas.mp4')

## 3. Transcribir con turbo, small y large

Esto cachea cada JSON en `evaluation/outputs/azzaro_whisper/transcripts/`. Si ya existe, no vuelve a transcribir.

In [3]:
resultados = {}
palabras = {}

for model_name in MODELOS:
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[model_name] = extraer_palabras(resultados[model_name], model_name)

pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras_modelo)}
    for modelo, palabras_modelo in palabras.items()
])

Transcribiendo / cargando cache: turbo
Transcribiendo / cargando cache: small
Transcribiendo / cargando cache: large


,modelo,palabras
0,turbo,2611
1,small,2590
2,large,2684


## 4. Detectar diferencias donde difieren los tres modelos

A partir de aca filtramos fuerte: solo quedan tramos donde `turbo`, `small` y `large` producen tres textos distintos.


In [4]:
def texto_modelo_en_rango(palabras_modelo, start, end, margen=0.20):
    seleccionadas = []
    left = max(float(start) - margen, 0.0)
    right = float(end) + margen
    for palabra in palabras_modelo:
        p_start = float(palabra.get("start", 0.0))
        p_end = float(palabra.get("end", p_start))
        centro = (p_start + p_end) / 2
        if left <= centro <= right or (p_start < right and p_end > left):
            seleccionadas.append(str(palabra.get("word", "")).strip())
    return " ".join(tok for tok in seleccionadas if tok).strip()


def unir_intervalos(intervalos, gap=0.75):
    intervalos = sorted(intervalos, key=lambda x: (x[0], x[1]))
    unidos = []
    for start, end in intervalos:
        if not unidos or start > unidos[-1][1] + gap:
            unidos.append([float(start), float(end)])
        else:
            unidos[-1][1] = max(unidos[-1][1], float(end))
    return unidos


def asegurar_mp4_browser(clip_path):
    # Reencodea a un MP4 conservador para que VSCode/Jupyter lo reproduzcan con audio.
    clip_path = Path(clip_path)
    tmp_path = clip_path.with_name(f"{clip_path.stem}.browser_tmp.mp4")
    cmd = [
        buscar_ffmpeg(),
        "-nostdin",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(clip_path),
        "-map",
        "0:v:0",
        "-map",
        "0:a:0?",
        "-c:v",
        "libx264",
        "-profile:v",
        "baseline",
        "-level",
        "3.0",
        "-pix_fmt",
        "yuv420p",
        "-preset",
        "veryfast",
        "-c:a",
        "aac",
        "-b:a",
        "128k",
        "-movflags",
        "+faststart",
        str(tmp_path),
    ]
    subprocess.run(cmd, check=True)
    tmp_path.replace(clip_path)
    return clip_path


# Primero se detectan todos los casos donde turbo, small y large dicen cosas distintas.
diferencias_por_par = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=5,
    )
    for diferencia in diferencias:
        diferencias_por_par.append({
            "start": float(diferencia["start"]),
            "end": float(diferencia["end"]),
            "comparacion": f"{modelo_a}_vs_{modelo_b}",
        })

intervalos_diff = [
    (d["start"], d["end"])
    for d in diferencias_por_par
    if d["end"] > d["start"]
]

filas_tres_modelos = []
for start, end in unir_intervalos(intervalos_diff):
    transcripciones_reales = {
        modelo: texto_modelo_en_rango(palabras[modelo], start, end)
        for modelo in MODELOS
    }
    transcripciones_norm = {
        modelo: normalizar_texto(texto)
        for modelo, texto in transcripciones_reales.items()
    }

    if all(transcripciones_norm.values()) and len(set(transcripciones_norm.values())) == len(MODELOS):
        filas_tres_modelos.append({
            "diff_id": len(filas_tres_modelos) + 1,
            "start": round(start, 3),
            "end": round(end, 3),
            "transcripcion_real_turbo": transcripciones_reales["turbo"],
            "transcripcion_real_small": transcripciones_reales["small"],
            "transcripcion_real_large": transcripciones_reales["large"],
            "norm_turbo": transcripciones_norm["turbo"],
            "norm_small": transcripciones_norm["small"],
            "norm_large": transcripciones_norm["large"],
        })

# Casos elegidos manualmente como los mas relevantes.
# Son los numeros de caso del notebook anterior, que empezaban en 0.
CASOS_ORIGINALES_RELEVANTES = [5, 6, 11, 12, 13, 22, 36, 45, 48, 64, 67, 68, 79]
DIFF_IDS_RELEVANTES = [caso + 1 for caso in CASOS_ORIGINALES_RELEVANTES]
orden_relevante = {diff_id: i + 1 for i, diff_id in enumerate(DIFF_IDS_RELEVANTES)}

filas_relevantes = [
    fila
    for fila in filas_tres_modelos
    if int(fila["diff_id"]) in orden_relevante
]
filas_relevantes.sort(key=lambda fila: orden_relevante[int(fila["diff_id"])])

clips_dir = WORKDIR / f"clips_transcripciones_reales_margen_{int(CLIP_MARGIN_SECONDS * 1000):04d}ms"
filas_relevantes = exportar_clips_diferencias(
    video_path,
    filas_relevantes,
    output_dir=clips_dir,
    margen=CLIP_MARGIN_SECONDS,
    max_clips=len(filas_relevantes),
)

for fila in filas_relevantes:
    fila["clip_path"] = str(asegurar_mp4_browser(fila["clip_path"]).resolve())

df_todos = pd.DataFrame(filas_tres_modelos)
df_tres = pd.DataFrame(filas_relevantes)

if not df_tres.empty:
    df_tres.insert(0, "caso_relevante", range(1, len(df_tres) + 1))
    df_tres["caso_original_notebook"] = df_tres["diff_id"].astype(int) - 1

faltantes = sorted(set(DIFF_IDS_RELEVANTES) - set(df_tres["diff_id"].astype(int))) if not df_tres.empty else DIFF_IDS_RELEVANTES
if faltantes:
    print(f"Atencion: faltan diff_id esperados: {faltantes}")

salida_relevantes = WORKDIR / "diferencias_tres_modelos_relevantes.csv"
df_tres.to_csv(salida_relevantes, index=False)

print(f"Casos donde difieren los tres modelos detectados: {len(df_todos)}")
print(f"Casos relevantes seleccionados: {len(df_tres)}")
print(f"CSV relevante: {salida_relevantes}")

cols = [
    "caso_relevante",
    "caso_original_notebook",
    "diff_id",
    "start",
    "end",
    "transcripcion_real_turbo",
    "transcripcion_real_small",
    "transcripcion_real_large",
    "clip_path",
]
display(df_tres[cols] if not df_tres.empty else df_tres)


Casos donde difieren los tres modelos detectados: 80
Casos relevantes seleccionados: 13
CSV relevante: C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\diferencias_tres_modelos_relevantes.csv


,caso_relevante,caso_original_notebook,diff_id,start,end,transcripcion_real_turbo,transcripcion_real_small,transcripcion_real_large,clip_path
0,1,5,6,142.70,143.28,ecuatoriano Duracán porque,ecuatoriano de Uracán porque,ecuatoriano... ...del huracán... ...porque,C:\Users\bianc\NLP Natural Language Processing...
1,2,6,7,172.36,174.66,y otro al repechaje. No fuiste ni al repechaje...,adentro y otro al repeteche no fuiste ni al re...,y otro alrededor. Y el repechaje no fuiste ni ...,C:\Users\bianc\NLP Natural Language Processing...
2,3,11,12,365.28,366.28,gol que era Maravilla. El gol,"gol que rama, el gol","que era Maravilla, el gol",C:\Users\bianc\NLP Natural Language Processing...
3,4,12,13,412.42,413.02,poner a Vergara refuerzo.,poner a bregar refuerzo,"poner a Berrán. Berrán, refuerzo.",C:\Users\bianc\NLP Natural Language Processing...
4,5,13,14,414.04,415.38,refuerzo. Con él ni suplente hoy de un,con Ignis suplente hoy de...,"Con él, mi suplente hoy de...",C:\Users\bianc\NLP Natural Language Processing...
5,6,22,23,525.00,526.30,"afuera con Caracas, la verduguiada que te","afuera con caraca, la verdurriada que te","afuera con Caracas, la verdugueada que te",C:\Users\bianc\NLP Natural Language Processing...
6,7,36,37,720.38,721.52,ganar la Libertadores. A menos de que después,ganar a la Libertadores a mente que después,ganar la Libertadores. A mente que después,C:\Users\bianc\NLP Natural Language Processing...
7,8,45,46,779.06,779.90,acá. Y estoy diciendo esto de Costas. Y estoy,acá y estoy siendo de Costa y estoy,y estoy diciendo esto de Kostas... ...y estoy ...,C:\Users\bianc\NLP Natural Language Processing...
8,9,48,49,807.20,809.22,Ya está. No hay excusas. Ahora armá tu equipo.,ya estaba no hay excusa ahora armad tu equipo,"Ya está, no hay excusa. Ahora armá tu equipo",C:\Users\bianc\NLP Natural Language Processing...
9,10,64,65,909.38,909.78,semestre. Dios me,semestre me libre,semestre. Dios me libre.,C:\Users\bianc\NLP Natural Language Processing...


## 5. Casos mas relevantes

Quedan solo los casos curados a mano. El texto mostrado es la transcripcion real del video completo en ese intervalo; el clip sirve para revisar ese momento puntual.


In [5]:
def video_mp4_embebido(clip_path):
    clip_path = Path(clip_path).resolve()
    data = base64.b64encode(clip_path.read_bytes()).decode("ascii")
    return HTML(f"""
    <video controls preload="metadata" playsinline style="width: 900px; max-width: 100%; background: #111;">
      <source src="data:video/mp4;base64,{data}" type="video/mp4">
      No se pudo reproducir este MP4 en el notebook.
    </video>
    """)


if df_tres.empty:
    print("No hay casos relevantes para mostrar.")
else:
    filas = df_tres if MAX_CASOS_A_MOSTRAR is None else df_tres.head(MAX_CASOS_A_MOSTRAR)
    for _, row in filas.iterrows():
        display(Markdown(
            f"### Caso relevante {int(row['caso_relevante'])} | "
            f"diff_{int(row['diff_id']):03d} | "
            f"transcripcion original {row['start']}s-{row['end']}s"
        ))

        clip_path = Path(row["clip_path"]).resolve()
        display(video_mp4_embebido(clip_path))
        print(str(clip_path))

        print("turbo:", row["transcripcion_real_turbo"])
        print("small:", row["transcripcion_real_small"])
        print("large:", row["transcripcion_real_large"])


### Caso relevante 1 | diff_006 | transcripcion original 142.7s-143.28s

C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\clips_transcripciones_reales_margen_0000ms\diff_006.mp4
turbo: ecuatoriano Duracán porque
small: ecuatoriano de Uracán porque
large: ecuatoriano... ...del huracán... ...porque


### Caso relevante 2 | diff_007 | transcripcion original 172.36s-174.66s

C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\clips_transcripciones_reales_margen_0000ms\diff_007.mp4
turbo: y otro al repechaje. No fuiste ni al repechaje con
small: adentro y otro al repeteche no fuiste ni al repeteche con
large: y otro alrededor. Y el repechaje no fuiste ni al repechaje con un


### Caso relevante 3 | diff_012 | transcripcion original 365.28s-366.28s

...

C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\clips_transcripciones_reales_margen_0000ms\diff_068.mp4
turbo: juego de Racing. Dios.
small: juego de race, sin Dios
large: de Racing, Dios.


### Caso relevante 12 | diff_069 | transcripcion original 943.74s-950.14s

C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\clips_transcripciones_reales_margen_0000ms\diff_069.mp4
turbo: juego de Racing. Y por qué rara maravilla la
small: juego de race me gole que ramaravilla la
large: juego de Racing. El gol que era maravilla la puta


### Caso relevante 13 | diff_080 | transcripcion original 1073.76s-1075.76s

C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\azzaro_whisper\clips_transcripciones_reales_margen_0000ms\diff_080.mp4
turbo: Y bueno. Y bueno. Se viene,
small: digo, si viene no
large: Y bueno, se viene, no


In [6]:
# Opcional: guardar una decision manual para un caso relevante puntual.
CASO_RELEVANTE = 1
MODELO_CORRECTO = ""  # opciones: "turbo", "small", "large", "ninguno", "empate"
NOTA = ""

revision_path = WORKDIR / "revision_manual_tres_modelos_relevantes.csv"

if df_tres.empty:
    print("No hay casos relevantes para guardar.")
else:
    if revision_path.exists():
        revision_tres = pd.read_csv(revision_path)
    else:
        revision_tres = df_tres.copy()
        revision_tres["modelo_correcto"] = ""
        revision_tres["nota"] = ""

    if MODELO_CORRECTO:
        mask = revision_tres["caso_relevante"].eq(CASO_RELEVANTE)
        revision_tres.loc[mask, "modelo_correcto"] = MODELO_CORRECTO
        revision_tres.loc[mask, "nota"] = NOTA
        revision_tres.to_csv(revision_path, index=False)
        display(revision_tres.loc[mask, ["caso_relevante", "transcripcion_real_turbo", "transcripcion_real_small", "transcripcion_real_large", "modelo_correcto", "nota"]])
        print(f"Guardado en: {revision_path}")
    else:
        completadas = revision_tres["modelo_correcto"].fillna("").ne("").sum()
        print(f"Completa MODELO_CORRECTO para guardar. Revisadas: {completadas} / {len(revision_tres)}")
        display(revision_tres[["caso_relevante", "transcripcion_real_turbo", "transcripcion_real_small", "transcripcion_real_large", "modelo_correcto", "nota"]])


Completa MODELO_CORRECTO para guardar. Revisadas: 0 / 13


,caso_relevante,transcripcion_real_turbo,transcripcion_real_small,transcripcion_real_large,modelo_correcto,nota
0,1,ecuatoriano Duracán porque,ecuatoriano de Uracán porque,ecuatoriano... ...del huracán... ...porque,,
1,2,y otro al repechaje. No fuiste ni al repechaje...,adentro y otro al repeteche no fuiste ni al re...,y otro alrededor. Y el repechaje no fuiste ni ...,,
2,3,gol que era Maravilla. El gol,"gol que rama, el gol","que era Maravilla, el gol",,
3,4,poner a Vergara refuerzo.,poner a bregar refuerzo,"poner a Berrán. Berrán, refuerzo.",,
4,5,refuerzo. Con él ni suplente hoy de un,con Ignis suplente hoy de...,"Con él, mi suplente hoy de...",,
5,6,"afuera con Caracas, la verduguiada que te","afuera con caraca, la verdurriada que te","afuera con Caracas, la verdugueada que te",,
6,7,ganar la Libertadores. A menos de que después,ganar a la Libertadores a mente que después,ganar la Libertadores. A mente que después,,
7,8,acá. Y estoy diciendo esto de Costas. Y estoy,acá y estoy siendo de Costa y estoy,y estoy diciendo esto de Kostas... ...y estoy ...,,
8,9,Ya está. No hay excusas. Ahora armá tu equipo.,ya estaba no hay excusa ahora armad tu equipo,"Ya está, no hay excusa. Ahora armá tu equipo",,
9,10,semestre. Dios me,semestre me libre,semestre. Dios me libre.,,
